<a href="https://colab.research.google.com/github/vinodhpatil/AgenticAIWork/blob/main/AgenticAIDay5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import time

# Configure API key
genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

# Discover available models
available_models = []
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        available_models.append(m.name)

print("Available models:", available_models)

# Select first available model
if not available_models:
    raise Exception("No compatible Gemini models available for this API key")

model_name = available_models[0]
print("Using model:", model_name)

model = genai.GenerativeModel(model_name)

# Agent function with retry
def simple_agent(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            print("Error:", e)

            wait = 2 ** attempt
            print(f"Retrying in {wait}s...")
            time.sleep(wait)

    return "Failed after retries"

# Prompt
question = "explain quantum physics"

answer = simple_agent(question)

print("Q:", question)
print("A:", answer)

Available models: ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-2.0-flash', 'models/gemini-2.0-flash-001', 'models/gemini-2.0-flash-lite-001', 'models/gemini-2.0-flash-lite', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-2.5-pro-preview-tts', 'models/gemma-3-1b-it', 'models/gemma-3-4b-it', 'models/gemma-3-12b-it', 'models/gemma-3-27b-it', 'models/gemma-3n-e4b-it', 'models/gemma-3n-e2b-it', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-pro-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-lite-preview-09-2025', 'models/gemini-3-pro-preview', 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview', 'models/gemini-3.1-pro-preview-customtools', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3-pro-image-preview', 'models/nano-banana-pro-preview', 'models/gemini-3.1-flash-image-preview', 'models/gemini-robotics-er-1.5-preview', 'models/gemini-2.5-computer-use-prev

In [ ]:
#step1- install all required packages
# langchain - core framework
# langchain-google-genai - integrate with gemini models
# langchain community - doc loaders, utilities
# python-doenv - load env variables
# pypdf and doc2txt - read pdf and word
# langchain-text-splitters - split text into chunks
# langchain-classic -LLM chain
# langchain-core - prompt template

!pip install -q langchain langchain-google-genai python-dotenv pypdf doc2txt langchain-text-splitters langchain-community langchain-classic langchain-core

#Step2 - import libraries
# import as - interact with env variables
# files - upload files
# userdata - access secrets stored in colabs

import os
from google.colab import files, userdata

#LLM wrapper from google
from langchain_google_genai import ChatGoogleGenerativeAI

#Libraries to allow doc of diff formats
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

#Library to split content into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# helps to create structure prompt
from langchain_core.prompts import PromptTemplate

# all steps defined above
from langchain_classic.chains import LLMChain

# conver output into usable format
from langchain_core.output_parsers import StrOutputParser

#Step3 - API key setup
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

#step4 - LLM and output parser
#temperature 0 - 0.5 - consistent response
#temperature 0.5 - 5  - creative response

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.8)
output_parser = StrOutputParser()

#step 5 - Create prompt
template = """

You are going to work as AI resume screener

Job Description:
{job_description}

Resume Text:
{resume_text}

Give a simple analysis:
- FIT Score (0-100)
- Top 5 matching skills
- Missing important skills
- One line verdict
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["job_description", "resume_text"],
    output_parser=output_parser
)

# combining all above perdefined steps so that it can work together (chains)
chain = LLMChain(llm=llm, prompt=prompt, output_parser=output_parser)

# step 6: upload and extract text out of it (chunks)
print("Please upload your resume (PDF/DOC/TXT:)")
uploaded = files.upload()

if not uploaded:
  raise ValueError("No files uploaded.")

# extract format
resume_path = list(uploaded.keys())[0]

if resume_path.endswith(".pdf"):
    loader = PyPDFLoader(resume_path)
elif resume_path.lower().endswith(".docx"):
    loader = Docx2txtLoader(resume_path)
elif resume_path.lower().endswith(".rtf"):
    loader = Docx2txtLoader(resume_path)
else:
  loader = TextLoader(resume_path, encoding="utf-8") # Fixed typo here as well (utp-8 -> utf-8)

docs = loader.load()

#split resume(document) into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs) # Renamed 'chunk' to 'chunks' for consistency and clarity
resume_text = " ".join(c.page_content for c in chunks).strip()

#error handling
if not resume_text:
  raise ValueError("could not extract text from uploaded files")

#step 7: run analysis and define job desription

job_description = """ we are hiring to a lead python developer with experience in python, AI, SQL, Cloud(AWS/GCP), and data analysis."""

result = chain.run(job_description=job_description, resume_text=resume_text)
print("******result Analysis*****")
print(result)

Please upload your resume (PDF/DOC/TXT:)


Saving Resume-VHP-Jan26.pdf to Resume-VHP-Jan26 (1).pdf
******result Analysis*****
Here's an analysis of the resume against the Lead Python Developer job description:

**FIT Score (0-100):** 88/100

**Top 5 matching skills:**
1.  **Leadership & Program Management:** The candidate has extensive experience in leading teams, managing complex programs, and driving strategic initiatives across various organizations.
2.  **Data Analysis & Data Science/ML:** Strong background in BI, data engineering, visualization (PowerBI, Azure Data Factory), and recent certification, patent, and hackathon success in Data Science/ML, with practical application in projects.
3.  **Cloud Technologies (Azure & AWS):** Demonstrates significant experience with Azure (6+ years) and recent project-level experience with AWS, covering key cloud components.
4.  **SQL/RDBMS:** Very strong and extensive experience with various RDBMS (Oracle, MySQL, SQL-Server) and NoSQL databases.
5.  **AI/Machine Learning:** Proven thr